In [ ]:
# IL-only GPU AdamW (PTX, khong C++) - 1/4: GPU + mono (KHONG build C/C++)
import os, subprocess, glob, re
!nvidia-smi
r=subprocess.run('wget -q https://dot.net/v1/dotnet-install.sh -O /tmp/dotnet-install.sh && bash /tmp/dotnet-install.sh --channel 10.0 --install-dir /content/dotnet', shell=True, capture_output=True, text=True, timeout=1200)
print('dotnet install rc=', r.returncode, (r.stderr or '')[-400:])
v=subprocess.run('/content/dotnet/dotnet --version', shell=True, capture_output=True, text=True)
print((v.stdout or '').split('\n')[0])
print('KHONG import torch - duong IL-only doc lap')

In [ ]:
# 2/3: clone backend + symlink cudart/cublas/cuda(driver)
import os, subprocess, glob
d='/content/TokenVectorCuda'
subprocess.run('rm -rf '+d, shell=True)
r=subprocess.run('git clone --depth 1 --branch prototype/cuda-backend https://github.com/nguyenhungtran18/TokenVector.git '+d, shell=True, capture_output=True, text=True, timeout=600)
print('clone rc=', r.returncode)
print('CUDA backend HEAD', subprocess.check_output('git -C '+d+' rev-parse --short HEAD',shell=True).decode().strip())
libdir='/content/tkvlib'; os.makedirs(libdir, exist_ok=True)
def pick(pats):
    for pat in pats:
        found=[p for p in glob.glob(pat) if os.path.exists(os.path.realpath(p))]
        if found: return sorted(found)[0]
    return None
cudart=pick(['/usr/local/cuda*/lib64/libcudart.so','/usr/lib/x86_64-linux-gnu/libcudart.so*'])
cublas=pick(['/usr/local/cuda*/lib64/libcublas.so','/usr/lib/x86_64-linux-gnu/libcublas.so*'])
cudrv=pick(['/usr/lib64-nvidia/libcuda.so*','/usr/lib/x86_64-linux-gnu/libcuda.so*','/usr/local/cuda/compat/libcuda.so*'])
print('cudart=',cudart); print('cublas=',cublas); print('cuda driver=',cudrv)
for nm,tgt in [('cudart.dll',cudart),('cublas.dll',cublas),('cuda.dll',cudrv)]:
    if tgt is None: continue
    p=libdir+'/'+nm
    if os.path.lexists(p): os.remove(p)
    os.symlink(tgt,p); print('symlink',nm,'->',tgt)
os.environ['LD_LIBRARY_PATH']=libdir+':'+os.environ.get('LD_LIBRARY_PATH','')
print('LD_LIBRARY_PATH=',os.environ['LD_LIBRARY_PATH'][:160])

In [ ]:
# 3/4: chay bang .NET 10 thay mono
import subprocess, os, re, math
exe=d+'/Release/cuda_backend/examples/ilgpu_step1.exe'
print('exe exists=',os.path.exists(exe),' bytes=',os.path.getsize(exe) if os.path.exists(exe) else 0)
rcfg=os.path.dirname(exe)+'/ilgpu_step1.runtimeconfig.json'
open(rcfg,'w').write('{"runtimeOptions":{"tfm":"net10.0","framework":{"name":"Microsoft.NETCore.App","version":"10.0.0"}}}')
r=subprocess.run(['/content/dotnet/dotnet',exe], env=dict(os.environ), capture_output=True, text=True, timeout=300)
print('dotnet rc=',r.returncode)
print(r.stdout[:2000])
print('STDERR:',(r.stderr or '')[:1500])

In [ ]:
# 4/4: reference Python thuan cho cung mot buoc
import math
T,D,V=4,8,8
W1=[[0.2+0.01*(i+j) for j in range(D)] for i in range(D)]
W2=[[0.3+0.01*(i+j) for j in range(V)] for i in range(D)]
X=[[0.1+0.02*(t+j) for j in range(D)] for t in range(T)]
ids=[(t+1)%V for t in range(T)]
def gelu(x):
    z=0.79788456*(x+0.044715*x*x*x); e=math.exp(2*z); th=(e-1)/(e+1)
    return 0.5*x*(1+th)
def dgelu(x):
    z=0.79788456*(x+0.044715*x*x*x); e=math.exp(2*z); th=(e-1)/(e+1)
    return 0.5*(1+th)+0.5*x*(1-th*th)*0.79788456*(1+0.134145*x*x)
Z=[[sum(X[t][k]*W1[k][j] for k in range(D))+0.1 for j in range(D)] for t in range(T)]
H=[[gelu(Z[t][j]) for j in range(D)] for t in range(T)]
LG=[[sum(H[t][j]*W2[j][v] for j in range(D))+0.05 for v in range(V)] for t in range(T)]
P=[]
for t in range(T):
    mx=max(LG[t]); e=[math.exp(v-mx) for v in LG[t]]; z=sum(e); P.append([v/z for v in e])
loss=-sum(math.log(P[t][ids[t]]) for t in range(T))/T
dL=[[(P[t][v]-(1.0 if v==ids[t] else 0.0))/T for v in range(V)] for t in range(T)]
dW2=[[sum(H[t][i]*dL[t][j] for t in range(T)) for j in range(V)] for i in range(D)]
dH=[[sum(dL[t][v]*W2[j][v] for v in range(V)) for j in range(D)] for t in range(T)]
dZ=[[dH[t][j]*dgelu(Z[t][j]) for j in range(D)] for t in range(T)]
dW1=[[sum(X[t][i]*dZ[t][j] for t in range(T)) for j in range(D)] for i in range(D)]
lr,b1,b2,eps,wd,bc1,bc2=0.01,0.9,0.999,1e-8,0.01,0.1,0.001
def adamw(W,g):
    m=[[0.0]*len(W[0]) for _ in W]; v=[[0.0]*len(W[0]) for _ in W]
    for i in range(len(W)):
        for j in range(len(W[0])):
            gg=g[i][j]; mi=b1*m[i][j]+(1-b1)*gg; vi=b2*v[i][j]+(1-b2)*gg*gg
            W[i][j]=W[i][j]-lr*(mi/bc1)/(math.sqrt(vi/bc2)+eps)-lr*wd*W[i][j]
adamw(W1,dW1); adamw(W2,dW2)
def vals(tag): return [float(v) for v in re.findall(r'^'+tag+r'\s+\d+\s+([0-9.eE+-]+)$',r.stdout,re.M)]
g1,g2=vals('W1NEW'),vals('W2NEW')
r1=[W1[i][0] for i in range(8)]
r2=[W2[i][0] for i in range(8)]
il_loss=[float(v) for v in re.findall(r'^LOSS\s+([0-9.eE+-]+)$',r.stdout,re.M)]
e1=max(abs(a-b) for a,b in zip(g1,r1)) if len(g1)==8 else None
e2=max(abs(a-b) for a,b in zip(g2,r2)) if len(g2)==8 else None
print('loss IL=',il_loss[0] if il_loss else None,' ref=',loss,' err=',abs(il_loss[0]-loss) if il_loss else None)
print('W1/W2 update maxerr=',e1,e2)
print('KET LUAN:', 'OK' if il_loss and e1 is not None and e2 is not None and abs(il_loss[0]-loss)<2e-4 and e1<2e-4 and e2<2e-4 else 'XEM OUTPUT')